# Self-Healing Machine Learning (SHML)

Automated Failure Detection, Diagnosis, Recovery,
and Safety Validation for Credit-Default Prediction


# Phase C — Controlled Missingness and Recovery

## Research Objective

Evaluate whether SHML's existing median-imputation
strategy maintains or improves predictive performance
compared with a predefined zero-imputation benchmark.

## Research Hypotheses

H1: Increasing BILL_AMT1 missingness produces measurable
predictive degradation under at least some conditions.

H2: Recovery effectiveness differs between imputation
strategies.

H4: The magnitude of performance changes depends on
missingness severity.

## Experimental Configuration

- Dataset: UCI Default of Credit Card Clients
- Model: Existing Logistic Regression baseline
- Corrupted feature: BILL_AMT1 only
- Missingness: 0%, 5%, 15%, 30%, 60%
- Random seed: 42 for the initial experiment
- Classification threshold: 0.50
- Healthy reference: Uncorrupted validation data
- Benchmark: Zero imputation
- SHML recovery: Training-median imputation
- Evaluation: F1, precision, recall, accuracy, ROC-AUC

## Experimental Controls

The model, training data, validation observations,
random missingness masks, and classification threshold
remain fixed between recovery strategies.

The test dataset remains untouched.

This initial experiment is exploratory. Statistical
validation across additional seeds follows in Phase D.

The UCI workbook contains no missing values. The experiment deliberately masks `BILL_AMT1` in copies of the validation features.


In [14]:

import numpy as np


def inject_bill_amt1_missingness(
    X_batch,
    missingness_rate,
    seed=42
):
    """
    Inject MCAR missingness into BILL_AMT1 only.

    Returns:
        corrupted DataFrame
        Boolean mask of modified rows
    """

    if not 0 <= missingness_rate <= 1:
        raise ValueError(
            "Missingness rate must be between 0 and 1."
        )

    if "BILL_AMT1" not in X_batch.columns:
        raise ValueError("BILL_AMT1 is missing.")

    X_corrupted = X_batch.copy(deep=True)

    rng = np.random.default_rng(seed)

    n_missing = int(
        len(X_corrupted) * missingness_rate
    )

    selected_positions = rng.choice(
        len(X_corrupted),
        size=n_missing,
        replace=False
    )

    mask = np.zeros(
        len(X_corrupted),
        dtype=bool
    )

    mask[selected_positions] = True

    X_corrupted.iloc[
        selected_positions,
        X_corrupted.columns.get_loc("BILL_AMT1")
    ] = np.nan

    return X_corrupted, mask


In [15]:
import pandas as pd

# Lightweight function test.
# No model training or dataset loading needed.
sample = pd.DataFrame({
    "BILL_AMT1": range(100),
    "BILL_AMT2": range(100, 200)
})

corrupted, mask = inject_bill_amt1_missingness(
    sample,
    missingness_rate=0.15,
    seed=42
)

assert corrupted["BILL_AMT1"].isna().sum() == 15
assert corrupted["BILL_AMT2"].equals(sample["BILL_AMT2"])
assert sample.isna().sum().sum() == 0
assert mask.sum() == 15

print("Phase C missingness injection test passed!")

Phase C missingness injection test passed!


In [16]:

from pathlib import Path
import sys

cwd = Path.cwd().resolve()
project_root = next(
    (path for path in (cwd, *cwd.parents)
     if (path / "src" / "run_pipeline.py").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Start the notebook kernel inside the SHML repository.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.detection import detect_feature_missingness
from src.diagnosis import diagnose_failure
from src.recovery import recover_missingness
from src.evaluation import evaluate_model
from src.safety import evaluate_recovery_safety


def compare_recovery_strategies(
    X_val,
    y_val,
    X_train,
    baseline_model,
    scaler,
    missingness_rate,
    seed=42
):
    """
    Compare a zero-fill benchmark with SHML median recovery.

    Reuses the existing trained model and SHML modules.
    """

    # 1. Inject missingness
    X_corrupted, mask = inject_bill_amt1_missingness(
        X_val,
        missingness_rate,
        seed
    )

    # 2. Healthy reference
    healthy_metrics = evaluate_model(
        X_batch=X_val,
        y_true=y_val,
        model=baseline_model,
        scaler=scaler
    )

    # 3. Zero-imputation benchmark
    X_zero = X_corrupted.copy(deep=True)
    X_zero["BILL_AMT1"] = (
        X_zero["BILL_AMT1"].fillna(0)
    )

    zero_metrics = evaluate_model(
        X_batch=X_zero,
        y_true=y_val,
        model=baseline_model,
        scaler=scaler
    )

    # 4. Existing SHML detection
    detection = detect_feature_missingness(
        X_batch=X_corrupted,
        feature="BILL_AMT1",
        reference_missingness=0.0,
        threshold=0.05
    )

    # 5. Existing SHML diagnosis
    diagnosis = diagnose_failure(detection)

    # 6. Existing SHML recovery
    training_median = X_train["BILL_AMT1"].median()

    X_recovered, recovery_log = recover_missingness(
        X_batch=X_corrupted,
        diagnosis=diagnosis,
        training_median=training_median
    )

    # 7. Evaluate SHML recovery
    recovered_metrics = evaluate_model(
        X_batch=X_recovered,
        y_true=y_val,
        model=baseline_model,
        scaler=scaler
    )

    # 8. Verify recovery integrity
    integrity_passed = (
        X_recovered.isna().sum().sum() == 0
        and X_recovered.shape == X_corrupted.shape
        and X_recovered.columns.equals(X_corrupted.columns)
        and X_recovered.loc[
            ~mask, "BILL_AMT1"
        ].equals(
            X_corrupted.loc[~mask, "BILL_AMT1"]
        )
        and X_recovered.drop(
            columns=["BILL_AMT1"]
        ).equals(
            X_corrupted.drop(columns=["BILL_AMT1"])
        )
        and (
            X_recovered.loc[mask, "BILL_AMT1"]
            == training_median
        ).all()
    )

    # 9. Existing SHML safety mechanism
    safety = evaluate_recovery_safety(
        recovered_metrics=recovered_metrics,
        healthy_metrics=healthy_metrics,
        data_integrity_passed=integrity_passed,
        max_f1_loss=0.10,
        max_recall_loss=0.10
    )

    # 10. Collect paired results
    return {
        "Missingness": missingness_rate,
        "Seed": seed,
        "Values_Missing": int(mask.sum()),
        "Detected": detection["failure_detected"],
        "Healthy_F1": healthy_metrics["f1"],
        "Zero_F1": zero_metrics["f1"],
        "SHML_F1": recovered_metrics["f1"],
        "Zero_Recall": zero_metrics["recall"],
        "SHML_Recall": recovered_metrics["recall"],
        "Zero_ROC_AUC": zero_metrics["roc_auc"],
        "SHML_ROC_AUC": recovered_metrics["roc_auc"],
        "F1_Improvement": (
            recovered_metrics["f1"] -
            zero_metrics["f1"]
        ),
        "Integrity_Passed": integrity_passed,
        "Safety_Decision": safety["decision"],
        "Safety_Reason": safety["reason"]
    }


In [17]:

from pathlib import Path
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Locate project root
cwd = Path.cwd().resolve()

PROJECT_ROOT = next(
    p for p in (cwd, *cwd.parents)
    if (p / "src" / "run_pipeline.py").exists()
)

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "working"
    / "default of credit card clients.xls"
)

# Load original dataset
df = pd.read_excel(
    DATA_PATH,
    header=1,
    engine="xlrd"
)

target = "default payment next month"

X = df.drop(columns=["ID", target])
y = df[target]

# Reuse original stratified splits
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

# Original preprocessing and model
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

baseline_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

baseline_model.fit(X_train_scaled, y_train)

print("Existing SHML baseline initialized.")
print("Training:", len(X_train))
print("Validation:", len(X_val))
print("Testing:", len(X_test))
print(
    "Training median:",
    X_train["BILL_AMT1"].median()
)


Existing SHML baseline initialized.
Training: 21000
Validation: 4500
Testing: 4500
Training median: 22225.5


In [18]:
missingness_levels = [0.00, 0.05, 0.15, 0.30, 0.60]

experiment_results = []

for rate in missingness_levels:
    result = compare_recovery_strategies(
        X_val=X_val,
        y_val=y_val,
        X_train=X_train,
        baseline_model=baseline_model,
        scaler=scaler,
        missingness_rate=rate,
        seed=42
    )

    experiment_results.append(result)

results_df = pd.DataFrame(experiment_results)

display(results_df.round(6))

,Missingness,Seed,Values_Missing,Detected,Healthy_F1,Zero_F1,SHML_F1,Zero_Recall,SHML_Recall,Zero_ROC_AUC,SHML_ROC_AUC,F1_Improvement,Integrity_Passed,Safety_Decision,Safety_Reason
0,0.00,42,0,False,0.356983,0.356983,0.356983,0.240201,0.240201,0.713025,0.713025,0.000000,True,ACCEPT,All checks passed
1,0.05,42,225,False,0.356983,0.361338,0.358898,0.244221,0.242211,0.713555,0.713210,-0.002440,True,ACCEPT,All checks passed
2,0.15,42,675,True,0.356983,0.366446,0.360947,0.250251,0.245226,0.708272,0.708958,-0.005499,True,ACCEPT,All checks passed
3,0.30,42,1350,True,0.356983,0.384116,0.373723,0.267337,0.257286,0.710521,0.711135,-0.010393,True,ACCEPT,All checks passed
4,0.60,42,2700,True,0.356983,0.398017,0.384949,0.282412,0.267337,0.704881,0.706199,-0.013068,True,ACCEPT,All checks passed



### Phase C — Initial Experimental Findings

Controlled BILL_AMT1 missingness was evaluated at
0%, 5%, 15%, 30%, and 60% using random seed 42.

Two strategies were compared:
1. Zero imputation (benchmark)
2. Training-median imputation (existing SHML recovery)

#### Finding C.1 — F1 Performance

Both strategies maintained or exceeded healthy
baseline F1 at every tested missingness level.

Zero imputation outperformed SHML median imputation
in F1 at every nonzero missingness level.

This does not support the superiority of median
imputation under the current configuration.

#### Finding C.2 — ROC-AUC Performance

At 60% missingness, ROC-AUC decreased from
0.713025 (healthy) to:

- Zero imputation: 0.704881
- SHML median imputation: 0.706199

Thus, increasing F1 did not necessarily correspond
to improved discrimination across thresholds.

#### Finding C.3 — Safety Decisions

The existing safety mechanism accepted every
SHML recovery candidate.

These decisions were consistent with the configured
F1, recall, and data-integrity requirements.

However, the safety mechanism does not compare
recovery candidates against alternative strategies.

#### Research Limitations

- Results currently use only random seed 42.
- Zero imputation is a benchmark intervention,
  not an untreated corrupted-data condition.
- Statistical significance has not been assessed.
- Alternative imputation strategies remain untested.
- Results do not establish diagnosis-guided
  recovery superiority.

These findings are preliminary and require
repeated-seed validation.


In [19]:

# Phase C — Step 4: Repeated Injection Seeds

import pandas as pd

seeds = range(20)
missingness_levels = [0.00, 0.05, 0.15, 0.30, 0.60]

repeated_results = []

for seed in seeds:
    for rate in missingness_levels:
        result = compare_recovery_strategies(
            X_val=X_val,
            y_val=y_val,
            X_train=X_train,
            baseline_model=baseline_model,
            scaler=scaler,
            missingness_rate=rate,
            seed=seed
        )
        repeated_results.append(result)

repeated_df = pd.DataFrame(repeated_results)

summary_df = (
    repeated_df
    .groupby("Missingness")
    .agg(
        Zero_F1_Mean=("Zero_F1", "mean"),
        SHML_F1_Mean=("SHML_F1", "mean"),
        F1_Difference_Mean=("F1_Improvement", "mean"),
        F1_Difference_SD=("F1_Improvement", "std"),
        Zero_AUC_Mean=("Zero_ROC_AUC", "mean"),
        SHML_AUC_Mean=("SHML_ROC_AUC", "mean")
    )
    .reset_index()
)

display(summary_df.round(6))


,Missingness,Zero_F1_Mean,SHML_F1_Mean,F1_Difference_Mean,F1_Difference_SD,Zero_AUC_Mean,SHML_AUC_Mean
0,0.00,0.356983,0.356983,0.000000,0.000000,0.713025,0.713025
1,0.05,0.360792,0.359251,-0.001541,0.001893,0.711718,0.712009
2,0.15,0.368320,0.364761,-0.003559,0.002160,0.710679,0.711232
3,0.30,0.377667,0.368637,-0.009030,0.003366,0.708555,0.709558
4,0.60,0.397203,0.382537,-0.014666,0.003712,0.704121,0.705421


In [20]:

# Phase C — Step 5: Paired 95% Confidence Intervals

import numpy as np
import pandas as pd
from scipy import stats

ci_results = []

for rate, group in repeated_df.groupby("Missingness"):

    # Paired difference for each injection seed
    differences = group["F1_Improvement"].to_numpy()

    n = len(differences)
    mean_diff = differences.mean()
    sd = differences.std(ddof=1)

    if np.isclose(sd, 0):
        ci_lower = mean_diff
        ci_upper = mean_diff
    else:
        se = sd / np.sqrt(n)

        margin = stats.t.ppf(
            0.975,
            df=n - 1
        ) * se

        ci_lower = mean_diff - margin
        ci_upper = mean_diff + margin

    ci_results.append({
        "Missingness": rate,
        "N_Seeds": n,
        "Mean_F1_Difference": mean_diff,
        "CI_Lower": ci_lower,
        "CI_Upper": ci_upper,
        "Seeds_SHML_Better": int(
            (differences > 0).sum()
        ),
        "Seeds_Zero_Better": int(
            (differences < 0).sum()
        ),
        "Seeds_Tied": int(
            (differences == 0).sum()
        )
    })

ci_df = pd.DataFrame(ci_results)

display(ci_df.round(6))


,Missingness,N_Seeds,Mean_F1_Difference,CI_Lower,CI_Upper,Seeds_SHML_Better,Seeds_Zero_Better,Seeds_Tied
0,0.00,20,0.000000,0.000000,0.000000,0,0,20
1,0.05,20,-0.001541,-0.002427,-0.000655,6,13,1
2,0.15,20,-0.003559,-0.004570,-0.002548,1,19,0
3,0.30,20,-0.009030,-0.010605,-0.007454,0,20,0
4,0.60,20,-0.014666,-0.016403,-0.012929,0,20,0



### Observation C.4 — Repeated Missingness Experiments

The recovery comparison was repeated using 20 random
missingness injection seeds.

Across all nonzero missingness levels, zero imputation
achieved a higher mean F1-score than SHML
training-median imputation.

The mean F1 difference (SHML minus zero) became
more negative as missingness increased:

- 5%:  -0.001541
- 15%: -0.003559
- 30%: -0.009030
- 60%: -0.014666

However, SHML median imputation achieved a slightly
higher mean ROC-AUC at each nonzero missingness level.

This demonstrates a trade-off between F1 performance
at the fixed classification threshold and overall
ranking discrimination.

The results suggest that the existing SHML recovery
strategy should not automatically be considered
superior to simple benchmark imputation.

Uncertainty analysis and additional recovery
strategies remain necessary.



### Observation C.5 — Paired Statistical Validation

Twenty controlled missingness experiments were performed
for each severity level using different random seeds.

The comparison was paired: both imputation strategies
used identical missingness masks within each seed.

#### Main Findings

At all nonzero missingness levels, zero imputation
achieved a higher mean F1-score than the existing
SHML median-imputation strategy.

At 60% missingness:
- Mean F1 difference (SHML - Zero): -0.014666
- 95% interval: [-0.016403, -0.012929]
- Zero imputation outperformed SHML in 20/20 seeds.

All nonzero-missingness intervals excluded zero.

#### Interpretation

The existing SHML median-imputation strategy does not
demonstrate superior F1 performance against the
zero-imputation benchmark.

The observed difference generally becomes larger
as missingness severity increases.

These findings support investigating alternative
recovery strategies rather than assuming that
median imputation is the optimal correction.

#### Statistical Limitations

The confidence intervals characterize variability
across missingness injections for a fixed model and
validation dataset.

They do not establish population-level uncertainty
or generalization to unseen datasets.

Results should be interpreted as conditional on
the experimental configuration.



### Observation C.5 — Paired Statistical Validation

Twenty controlled missingness experiments were performed
for each severity level using different random seeds.

The comparison was paired: both imputation strategies
used identical missingness masks within each seed.

#### Main Findings

At all nonzero missingness levels, zero imputation
achieved a higher mean F1-score than the existing
SHML median-imputation strategy.

At 60% missingness:
- Mean F1 difference (SHML - Zero): -0.014666
- 95% interval: [-0.016403, -0.012929]
- Zero imputation outperformed SHML in 20/20 seeds.

All nonzero-missingness intervals excluded zero.

#### Interpretation

The existing SHML median-imputation strategy does not
demonstrate superior F1 performance against the
zero-imputation benchmark.

The observed difference generally becomes larger
as missingness severity increases.

These findings support investigating alternative
recovery strategies rather than assuming that
median imputation is the optimal correction.

#### Statistical Limitations

The confidence intervals characterize variability
across missingness injections for a fixed model and
validation dataset.

They do not establish population-level uncertainty
or generalization to unseen datasets.

Results should be interpreted as conditional on
the experimental configuration.


In [21]:
from pathlib import Path

cwd = Path.cwd().resolve()

PROJECT_ROOT = next(
    p for p in (cwd, *cwd.parents)
    if (p / "src" / "run_pipeline.py").exists()
)

FIGURES_DIR = (
    PROJECT_ROOT
    / "results"
    / "research"
    / "bill_amt1"
    / "figures"
)

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Figures will be saved to:")
print(FIGURES_DIR)

Figures will be saved to:
/Users/divya/Desktop/School/IU/GitStuff/SHML/results/research/bill_amt1/figures


In [22]:
import plotly.graph_objects as go

def show_research_figure(fig, filename):
    path = FIGURES_DIR / filename
    fig.write_html(path, include_plotlyjs="cdn")
    fig.show(renderer="plotly_mimetype")
    print("Saved:", path)


def recovery_lines(data, series, baseline, title, metric):
    x = data["Missingness"]
    fig = go.Figure()
    for column, label, symbol in series:
        fig.add_scatter(
            x=x, y=data[column], mode="lines+markers", name=label, marker_symbol=symbol,
            hovertemplate="%{fullData.name}: %{y:.4f}<extra></extra>",
        )
    fig.add_scatter(
        x=x, y=[baseline] * len(x), mode="lines", line=dict(dash="dash", color="gray"),
        name="Healthy baseline", hovertemplate="%{fullData.name}: %{y:.4f}<extra></extra>",
    )
    fig.update_layout(
        title=title,
        xaxis=dict(title="BILL_AMT1 missingness", tickvals=x, tickformat=".0%"),
        yaxis_title=metric, hovermode="x unified",
    )
    return fig


plot_df = summary_df.sort_values("Missingness")
fig = recovery_lines(
    plot_df,
    [("Zero_F1_Mean", "Zero imputation", "circle"),
     ("SHML_F1_Mean", "SHML median imputation", "square")],
    0.356983, "F1 vs. BILL_AMT1 Missingness", "Mean F1 score",
)
show_research_figure(fig, "f1_vs_missingness.html")


Saved: /Users/divya/Desktop/School/IU/GitStuff/SHML/results/research/bill_amt1/figures/f1_vs_missingness.html


In [23]:
plot_df = summary_df.sort_values("Missingness")
fig = recovery_lines(
    plot_df,
    [("Zero_AUC_Mean", "Zero imputation", "circle"),
     ("SHML_AUC_Mean", "SHML median imputation", "square")],
    0.713025, "ROC-AUC vs. BILL_AMT1 Missingness", "Mean ROC-AUC",
)
show_research_figure(fig, "roc_auc_vs_missingness.html")


Saved: /Users/divya/Desktop/School/IU/GitStuff/SHML/results/research/bill_amt1/figures/roc_auc_vs_missingness.html


In [24]:
plot_ci = ci_df.loc[ci_df["Missingness"] > 0].sort_values("Missingness")
mean_diff = plot_ci["Mean_F1_Difference"]
lower = plot_ci["CI_Lower"]
upper = plot_ci["CI_Upper"]

fig = go.Figure(go.Scatter(
    x=plot_ci["Missingness"], y=mean_diff, mode="lines+markers",
    error_y=dict(type="data", symmetric=False, array=upper - mean_diff,
                 arrayminus=mean_diff - lower, width=5),
    customdata=plot_ci[["CI_Lower", "CI_Upper"]],
    hovertemplate="Missingness: %{x:.0%}<br>Mean difference: %{y:+.4f}"
                  "<br>95% CI: [%{customdata[0]:+.4f}, %{customdata[1]:+.4f}]<extra></extra>",
))
fig.add_hline(y=0, line_color="gray")
fig.update_layout(
    title="Mean F1 Difference with 95% Confidence Intervals",
    xaxis=dict(title="BILL_AMT1 missingness", tickvals=plot_ci["Missingness"], tickformat=".0%"),
    yaxis_title="Mean F1 difference (SHML − zero)",
)
show_research_figure(fig, "recovery_confidence_intervals.html")


Saved: /Users/divya/Desktop/School/IU/GitStuff/SHML/results/research/bill_amt1/figures/recovery_confidence_intervals.html


In [25]:
phase_c_table = summary_df.copy()
phase_c_table["Missingness"] = phase_c_table["Missingness"].apply(lambda x: f"{int(x * 100)}%")
phase_c_table = phase_c_table.round(6)

display(phase_c_table)

,Missingness,Zero_F1_Mean,SHML_F1_Mean,F1_Difference_Mean,F1_Difference_SD,Zero_AUC_Mean,SHML_AUC_Mean
0,0%,0.356983,0.356983,0.000000,0.000000,0.713025,0.713025
1,5%,0.360792,0.359251,-0.001541,0.001893,0.711718,0.712009
2,15%,0.368320,0.364761,-0.003559,0.002160,0.710679,0.711232
3,30%,0.377667,0.368637,-0.009030,0.003366,0.708555,0.709558
4,60%,0.397203,0.382537,-0.014666,0.003712,0.704121,0.705421


## Phase C — Visualization Summary

### Figure 1 — F1-score vs. Missingness Severity
Both zero imputation and SHML median imputation maintained or exceeded the healthy baseline F1-score as missingness increased. However, zero imputation achieved the higher mean F1-score at every nonzero missingness level.

### Figure 2 — ROC-AUC vs. Missingness Severity
Both recovery strategies showed declining ROC-AUC as missingness severity increased. SHML median imputation retained slightly higher mean ROC-AUC than zero imputation across nonzero missingness levels.

### Figure 3 — Mean F1 Difference with 95% Confidence Intervals
The plotted quantity is SHML F1 minus zero-imputation F1. Negative values across all nonzero missingness levels indicate that zero imputation outperformed SHML median imputation on mean F1-score. The difference became more negative as missingness severity increased.

## C.9 — Recovery Strategy Win-Count Analysis

In [26]:
win_df = ci_df.loc[ci_df["Missingness"] > 0].sort_values("Missingness")
fig = go.Figure()
for column, label in [
    ("Seeds_Zero_Better", "Zero imputation"),
    ("Seeds_SHML_Better", "SHML median imputation"),
    ("Seeds_Tied", "Ties"),
]:
    fig.add_bar(
        x=win_df["Missingness"].map("{:.0%}".format), y=win_df[column],
        name=label, texttemplate="%{y}", textposition="outside",
        hovertemplate="%{x} missing<br>%{fullData.name}: %{y} of 20 seeds<extra></extra>",
    )
fig.update_layout(
    title="Recovery Strategy F1 Wins Across 20 Seeds", barmode="group",
    xaxis_title="BILL_AMT1 missingness", yaxis=dict(title="Number of seeds", range=[0, 23]),
)
show_research_figure(fig, "recovery_strategy_win_counts.html")


Saved: /Users/divya/Desktop/School/IU/GitStuff/SHML/results/research/bill_amt1/figures/recovery_strategy_win_counts.html



### Observation C.9 — Recovery Strategy Win Counts

The experiment compared zero imputation and SHML median
imputation across 20 random missingness injection seeds.

At 5% missingness, zero imputation achieved higher F1
in 13 seeds, while SHML median imputation performed
better in 6 seeds, with 1 tie.

At 15% missingness, zero imputation performed better
in 19 of 20 seeds.

At 30% and 60% missingness, zero imputation achieved
higher F1 in all 20 seeds.

These results demonstrate that the observed F1 advantage
of zero imputation becomes more consistent as missingness
severity increases.

However, these findings are specific to the existing
Logistic Regression model, validation sample, and
controlled MCAR missingness experiment.


## Phase C — Step 8: Correlation-Guided Recovery

In [27]:

# Phase C — Step 8A
# Train a regression-based BILL_AMT1 imputer

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

imputation_model = LinearRegression()

# Fit ONLY on training data
imputation_model.fit(
    X_train[["BILL_AMT2"]],
    X_train["BILL_AMT1"]
)

# Inspect training fit (not held-out performance)
train_predictions = imputation_model.predict(
    X_train[["BILL_AMT2"]]
)

print("Regression Imputer Trained")
print("Coefficient:", imputation_model.coef_[0])
print("Intercept:", imputation_model.intercept_)

print(
    "Training MAE:",
    mean_absolute_error(
        X_train["BILL_AMT1"],
        train_predictions
    )
)

print(
    "Training R²:",
    r2_score(
        X_train["BILL_AMT1"],
        train_predictions
    )
)


Regression Imputer Trained
Coefficient: 0.985584374025496
Intercept: 2761.8103539393996
Training MAE: 8469.438904604587
Training R²: 0.9072685583323683


In [28]:

# Phase C — Step 8B
# Recover missing BILL_AMT1 using BILL_AMT2

def recover_bill_amt1_regression(
    X_corrupted,
    imputation_model
):
    X_repaired = X_corrupted.copy(deep=True)

    missing_mask = X_repaired["BILL_AMT1"].isna()

    if missing_mask.any():

        predicted_values = imputation_model.predict(
            X_repaired.loc[
                missing_mask, ["BILL_AMT2"]
            ]
        )

        X_repaired.loc[
            missing_mask, "BILL_AMT1"
        ] = predicted_values

    return X_repaired


In [29]:

# Phase C — Step 8C
# Compare zero, median, and regression imputation

regression_results = []

for seed in range(20):
    for rate in [0.00, 0.05, 0.15, 0.30, 0.60]:

        # Generate the same corruption mask
        X_corrupted, mask = inject_bill_amt1_missingness(
            X_val,
            missingness_rate=rate,
            seed=seed
        )

        # Recover using regression imputation
        X_regression = recover_bill_amt1_regression(
            X_corrupted,
            imputation_model
        )

        # Verify integrity
        assert not X_regression.isna().any().any()

        np.testing.assert_allclose(
            X_regression.loc[~mask, "BILL_AMT1"].to_numpy(),
            X_val.loc[~mask, "BILL_AMT1"].to_numpy(),
            rtol=0,
            atol=0
        )

        pd.testing.assert_frame_equal(
            X_regression.drop(columns=["BILL_AMT1"]),
            X_val.drop(columns=["BILL_AMT1"])
        )

        assert X_regression.loc[
            mask, "BILL_AMT1"
        ].notna().all()

        # Evaluate regression recovery
        metrics = evaluate_model(
            X_batch=X_regression,
            y_true=y_val,
            model=baseline_model,
            scaler=scaler
        )

        # Measure reconstruction error
        reconstruction_mae = (
            mean_absolute_error(
                X_val.loc[mask, "BILL_AMT1"],
                X_regression.loc[mask, "BILL_AMT1"]
            )
            if mask.any()
            else np.nan
        )

        regression_results.append({
            "Missingness": rate,
            "Seed": seed,
            "Regression_F1": metrics["f1"],
            "Regression_Recall": metrics["recall"],
            "Regression_AUC": metrics["roc_auc"],
            "Reconstruction_MAE": reconstruction_mae
        })

regression_df = pd.DataFrame(regression_results)

# Join on the exact same seeds and missingness levels
all_strategies_df = repeated_df.merge(
    regression_df,
    on=["Missingness", "Seed"],
    validate="one_to_one"
)

assert len(all_strategies_df) == 100

strategy_summary = (
    all_strategies_df
    .groupby("Missingness")
    .agg(
        Zero_F1=("Zero_F1", "mean"),
        Median_F1=("SHML_F1", "mean"),
        Regression_F1=("Regression_F1", "mean"),
        Zero_AUC=("Zero_ROC_AUC", "mean"),
        Median_AUC=("SHML_ROC_AUC", "mean"),
        Regression_AUC=("Regression_AUC", "mean"),
        Reconstruction_MAE=("Reconstruction_MAE", "mean")
    )
    .reset_index()
)

strategy_summary["Regression_vs_Zero_F1"] = (
    strategy_summary["Regression_F1"]
    - strategy_summary["Zero_F1"]
)

strategy_summary["Regression_vs_Median_F1"] = (
    strategy_summary["Regression_F1"]
    - strategy_summary["Median_F1"]
)

display(strategy_summary.round(6))


,Missingness,Zero_F1,Median_F1,Regression_F1,Zero_AUC,Median_AUC,Regression_AUC,Reconstruction_MAE,Regression_vs_Zero_F1,Regression_vs_Median_F1
0,0.00,0.356983,0.356983,0.356983,0.713025,0.713025,0.713025,NaN,0.000000,0.000000
1,0.05,0.360792,0.359251,0.356887,0.711718,0.712009,0.712957,8707.555847,-0.003905,-0.002364
2,0.15,0.368320,0.364761,0.357321,0.710679,0.711232,0.712912,8875.826240,-0.010999,-0.007440
3,0.30,0.377667,0.368637,0.357428,0.708555,0.709558,0.712699,8567.597733,-0.020239,-0.011209
4,0.60,0.397203,0.382537,0.357287,0.704121,0.705421,0.712546,8717.835319,-0.039916,-0.025250



### Observation C.8 — Correlation-Guided Recovery

A Linear Regression model was trained to estimate
BILL_AMT1 using BILL_AMT2.

The regression imputer was fitted using training data
only and evaluated under controlled BILL_AMT1 missingness
at 0%, 5%, 15%, 30%, and 60%.

Experiments were repeated across 20 random
missingness injection seeds.

#### Main Findings

Zero imputation achieved the highest mean F1-score
at every nonzero missingness level.

Regression imputation achieved the highest mean
ROC-AUC at every nonzero missingness level.

At 60% missingness:

- Healthy F1: 0.356983
- Zero F1: 0.397203
- Median F1: 0.382537
- Regression F1: 0.357287

- Healthy ROC-AUC: 0.713025
- Zero ROC-AUC: 0.704121
- Median ROC-AUC: 0.705421
- Regression ROC-AUC: 0.712546

Regression imputation preserved ROC-AUC more closely
to the healthy baseline than the other strategies.

#### Interpretation

The preferred recovery method depends on the
evaluation criterion.

A strategy producing the highest F1-score at a fixed
threshold may not preserve ranking discrimination
as effectively as an alternative method.

The strong correlation between BILL_AMT1 and BILL_AMT2
provides a plausible explanation for why regression
reconstruction preserves predictive performance.

This remains an exploratory finding rather than
proof of causal feature redundancy.

#### Limitations

Results are conditional on the existing fixed
Logistic Regression model and validation dataset.

The 20 seeds vary only the missingness injection.

The regression imputer is a candidate recovery method,
not yet integrated into automated SHML method selection.


## Phase C — Step 9: Statistical Evaluation of Three Recovery Strategies

In [30]:

# Phase C — Step 9A
# Paired comparison across three recovery strategies

import numpy as np
import pandas as pd
from scipy import stats

comparisons = {
    "Regression vs Zero": {
        "F1": ("Regression_F1", "Zero_F1"),
        "ROC-AUC": ("Regression_AUC", "Zero_ROC_AUC")
    },
    "Regression vs Median": {
        "F1": ("Regression_F1", "SHML_F1"),
        "ROC-AUC": ("Regression_AUC", "SHML_ROC_AUC")
    },
    "Median vs Zero": {
        "F1": ("SHML_F1", "Zero_F1"),
        "ROC-AUC": ("SHML_ROC_AUC", "Zero_ROC_AUC")
    }
}

evaluation_results = []

for rate, group in all_strategies_df.groupby("Missingness"):

    group = group.sort_values("Seed")

    for comparison_name, metrics in comparisons.items():

        for metric_name, (first, second) in metrics.items():

            differences = (
                group[first] - group[second]
            ).to_numpy()

            n = len(differences)
            mean_diff = differences.mean()
            sd = differences.std(ddof=1)

            if np.isclose(sd, 0):
                lower = mean_diff
                upper = mean_diff
            else:
                se = sd / np.sqrt(n)
                margin = stats.t.ppf(
                    0.975, df=n - 1
                ) * se

                lower = mean_diff - margin
                upper = mean_diff + margin

            evaluation_results.append({
                "Missingness": rate,
                "Comparison": comparison_name,
                "Metric": metric_name,
                "N_Seeds": n,
                "Mean_Difference": mean_diff,
                "CI_Lower": lower,
                "CI_Upper": upper,
                "First_Better": int(
                    (differences > 0).sum()
                ),
                "Second_Better": int(
                    (differences < 0).sum()
                ),
                "Ties": int(
                    (differences == 0).sum()
                )
            })

evaluation_df = pd.DataFrame(evaluation_results)

assert len(evaluation_df) == 30
assert evaluation_df["N_Seeds"].eq(20).all()

display(evaluation_df.round(6))


,Missingness,Comparison,Metric,N_Seeds,Mean_Difference,CI_Lower,CI_Upper,First_Better,Second_Better,Ties
0,0.00,Regression vs Zero,F1,20,0.000000,0.000000,0.000000,0,0,20
1,0.00,Regression vs Zero,ROC-AUC,20,0.000000,0.000000,0.000000,0,0,20
2,0.00,Regression vs Median,F1,20,0.000000,0.000000,0.000000,0,0,20
3,0.00,Regression vs Median,ROC-AUC,20,0.000000,0.000000,0.000000,0,0,20
4,0.00,Median vs Zero,F1,20,0.000000,0.000000,0.000000,0,0,20
5,0.00,Median vs Zero,ROC-AUC,20,0.000000,0.000000,0.000000,0,0,20
6,0.05,Regression vs Zero,F1,20,-0.003905,-0.004882,-0.002929,0,20,0
7,0.05,Regression vs Zero,ROC-AUC,20,0.001239,0.000758,0.001720,18,2,0
8,0.05,Regression vs Median,F1,20,-0.002364,-0.002969,-0.001760,0,20,0
9,0.05,Regression vs Median,ROC-AUC,20,0.000948,0.000480,0.001415,16,4,0


Each comparison is calculated as the first strategy minus the second. A positive difference favors the first strategy; a negative difference favors the second.

The intervals describe variation across injection seeds with the validation set and trained model held fixed. They do not quantify generalization uncertainty or correct for multiple comparisons.

In [31]:

# Focus on the regression recovery comparisons
regression_evaluation = evaluation_df[
    (evaluation_df["Comparison"].isin([
        "Regression vs Zero",
        "Regression vs Median"
    ]))
    & (evaluation_df["Missingness"] > 0)
].copy()

display(
    regression_evaluation[
        [
            "Missingness",
            "Comparison",
            "Metric",
            "Mean_Difference",
            "CI_Lower",
            "CI_Upper",
            "First_Better",
            "Second_Better",
            "Ties"
        ]
    ].round(6)
)


,Missingness,Comparison,Metric,Mean_Difference,CI_Lower,CI_Upper,First_Better,Second_Better,Ties
6,0.05,Regression vs Zero,F1,-0.003905,-0.004882,-0.002929,0,20,0
7,0.05,Regression vs Zero,ROC-AUC,0.001239,0.000758,0.001720,18,2,0
8,0.05,Regression vs Median,F1,-0.002364,-0.002969,-0.001760,0,20,0
9,0.05,Regression vs Median,ROC-AUC,0.000948,0.000480,0.001415,16,4,0
12,0.15,Regression vs Zero,F1,-0.010999,-0.012718,-0.009280,0,20,0
13,0.15,Regression vs Zero,ROC-AUC,0.002233,0.001513,0.002954,19,1,0
14,0.15,Regression vs Median,F1,-0.007440,-0.008879,-0.006001,0,20,0
15,0.15,Regression vs Median,ROC-AUC,0.001680,0.001167,0.002192,19,1,0
18,0.30,Regression vs Zero,F1,-0.020239,-0.022530,-0.017947,0,20,0
19,0.30,Regression vs Zero,ROC-AUC,0.004144,0.003416,0.004872,20,0,0


In [32]:
plot_df = strategy_summary.sort_values("Missingness")
fig = recovery_lines(
    plot_df,
    [("Zero_F1", "Zero imputation", "circle"),
     ("Median_F1", "SHML median imputation", "square"),
     ("Regression_F1", "Regression imputation", "triangle-up")],
    0.356983, "Recovery Strategy Comparison — F1 Across 20 Seeds", "Mean F1",
)
show_research_figure(fig, "three_strategy_f1_comparison.html")


In [33]:
plot_df = strategy_summary.sort_values("Missingness")
fig = recovery_lines(
    plot_df,
    [("Zero_AUC", "Zero imputation", "circle"),
     ("Median_AUC", "SHML median imputation", "square"),
     ("Regression_AUC", "Regression imputation", "triangle-up")],
    0.713025, "Recovery Strategy Comparison — ROC-AUC Across 20 Seeds", "Mean ROC-AUC",
)
show_research_figure(fig, "three_strategy_auc_comparison.html")


## Phase E — Safety Validation

### Step 10A: Harmful Recovery Candidate

In [34]:

# Step 10A — Create a stress-test repair candidate

from src.evaluation import evaluate_model
from src.safety import evaluate_recovery_safety

# Use a reproducible missingness scenario
X_corrupted, mask = inject_bill_amt1_missingness(
    X_val,
    missingness_rate=0.60,
    seed=42
)

# Create a deliberately harmful candidate
X_harmful = X_corrupted.copy(deep=True)

# Extreme, unrealistic replacement for stress testing
harmful_value = 1_000_000_000

X_harmful.loc[
    mask, "BILL_AMT1"
] = harmful_value

# Verify no missing values remain
assert not X_harmful.isna().any().any()

# Preserve all other features
pd.testing.assert_frame_equal(
    X_harmful.drop(columns=["BILL_AMT1"]),
    X_val.drop(columns=["BILL_AMT1"])
)

print("Harmful candidate created.")
print("Rows modified:", int(mask.sum()))
print("Replacement value:", harmful_value)


Harmful candidate created.
Rows modified: 2700
Replacement value: 1000000000


In [35]:

# Step 10B — Compare healthy and harmful performance

healthy_metrics = evaluate_model(
    X_batch=X_val,
    y_true=y_val,
    model=baseline_model,
    scaler=scaler
)

harmful_metrics = evaluate_model(
    X_batch=X_harmful,
    y_true=y_val,
    model=baseline_model,
    scaler=scaler
)

comparison_safety = pd.DataFrame({
    "Healthy": healthy_metrics,
    "Harmful_Candidate": harmful_metrics
})

comparison_safety["Difference"] = (
    comparison_safety["Harmful_Candidate"]
    - comparison_safety["Healthy"]
)

display(comparison_safety.round(6))


,Healthy,Harmful_Candidate,Difference
accuracy,0.808667,0.790444,-0.018222
precision,0.694767,0.732143,0.037375
recall,0.240201,0.082412,-0.157789
f1,0.356983,0.148148,-0.208835
roc_auc,0.713025,0.535436,-0.177589


In [36]:

# Step 10C — Evaluate candidate safety

# The candidate replaces only injected missing values,
# so structural integrity is preserved.
integrity_passed = (
    X_harmful.shape == X_val.shape
    and X_harmful.columns.equals(X_val.columns)
    and not X_harmful.isna().any().any()
)

safety_result = evaluate_recovery_safety(
    recovered_metrics=harmful_metrics,
    healthy_metrics=healthy_metrics,
    data_integrity_passed=integrity_passed,
    max_f1_loss=0.10,
    max_recall_loss=0.10
)

print("Safety decision:", safety_result["decision"])
print("Reason:", safety_result["reason"])
print("F1 loss (%):", round(safety_result["f1_loss_pct"], 4))
print("Recall loss (%):", round(safety_result["recall_loss_pct"], 4))


Safety decision: REJECT
Reason: F1 loss exceeds tolerance; Recall loss exceeds tolerance
F1 loss (%): 58.4999
Recall loss (%): 65.6904


In [37]:

# Step 10D — Simulated rollback decision

approved_batch = X_val.copy(deep=True)

# Select candidate only when accepted
if safety_result["decision"] == "ACCEPT":
    selected_batch = X_harmful.copy(deep=True)
else:
    selected_batch = approved_batch.copy(deep=True)

selected_metrics = evaluate_model(
    X_batch=selected_batch,
    y_true=y_val,
    model=baseline_model,
    scaler=scaler
)

if safety_result["decision"] == "REJECT":
    pd.testing.assert_frame_equal(
        selected_batch,
        approved_batch
    )

    print("Rejected candidate was not promoted.")
    print("Approved state preserved.")
else:
    print("Candidate accepted under existing safety rules.")

print("Selected F1:", round(selected_metrics["f1"], 6))
print("Healthy F1:", round(healthy_metrics["f1"], 6))


Rejected candidate was not promoted.
Approved state preserved.
Selected F1: 0.356983
Healthy F1: 0.356983


## Phase E — Observation E.1: Safety Rejection and Approved-State Preservation

### Research Objective

Evaluate whether the existing SHML safety controller can identify and reject a recovery candidate that causes measurable predictive degradation, while preserving the previously approved model-input state.

This experiment primarily investigates **H3 — Safety Validation**.

### Experimental Setup

- **Dataset:** UCI Default of Credit Card Clients
- **Baseline model:** Logistic Regression
- **Evaluation dataset:** Validation set (4,500 observations)
- **Corrupted feature:** `BILL_AMT1` only
- **Injected missingness:** 60% (2,700 observations)
- **Random seed:** 42
- **Harmful replacement value:** 1,000,000,000
- **Safety thresholds:** Maximum 10% relative loss in F1 and recall
- **Safety criteria:** Valid performance metrics, data integrity, F1 tolerance, and recall tolerance

The extreme replacement value was deliberately selected to stress-test the existing safety mechanism. It is not proposed as a realistic or appropriate recovery method.

### Experimental Results

| Metric | Healthy Baseline | Harmful Candidate | Difference |
|---|---:|---:|---:|
| Accuracy | 0.808667 | 0.790444 | −0.018222 |
| Precision | 0.694767 | 0.732143 | +0.037375 |
| Recall | 0.240201 | 0.082412 | −0.157789 |
| F1-score | 0.356983 | 0.148148 | −0.208835 |
| ROC-AUC | 0.713025 | 0.535436 | −0.177589 |

The harmful candidate produced substantial degradation in F1-score, recall, and ROC-AUC.

Although precision increased, the decrease in recall indicates that the candidate failed to identify a large proportion of actual defaults compared with the healthy baseline.

### Safety Evaluation

The existing safety controller returned:

**Decision: REJECT**

**Reasons:**
- F1 loss exceeded the configured tolerance.
- Recall loss exceeded the configured tolerance.

The measured relative losses were:

| Safety measure | Observed loss | Maximum permitted loss | Result |
|---|---:|---:|---|
| F1 | 58.50% | 10% | REJECT |
| Recall | 65.69% | 10% | REJECT |

Both losses exceeded the predefined safety tolerances.

### Simulated Rollback Verification

Following rejection, the experiment selected the previously approved healthy validation batch instead of the harmful candidate.

The following checks succeeded:

- The rejected candidate was not promoted.
- The selected batch matched the approved batch.
- The selected F1-score remained 0.356983.
- The approved state was preserved in memory.

This demonstrates successful **offline candidate rejection and approved-state preservation** under the tested conditions.

It does not demonstrate persistent model rollback, deployment restoration, or production-state management.

### Interpretation

The experiment provides evidence that the SHML safety controller can reject a candidate when its actual predictive performance violates predefined safety thresholds.

Unlike earlier acceptance-only experiments, this experiment demonstrates a rejection caused by measurable performance degradation on validation data.

The result strengthens the evidence supporting H3, although it does not establish safety robustness across all possible recovery failures.

### Limitations

1. The harmful replacement value is extreme and unrealistic.
2. Only one harmful candidate and missingness configuration were evaluated.
3. The rollback mechanism was simulated through in-memory candidate selection.
4. Structural integrity checks did not comprehensively assess semantic validity or plausibility of the repaired values.
5. The safety controller currently evaluates F1 and recall losses but does not impose a separate ROC-AUC safety threshold.

### Research Conclusion

**H3 is partially supported.**

The SHML framework successfully rejected a deliberately harmful recovery candidate and preserved the previously approved validation state in an offline simulation.

Further experiments are required to establish whether the safety mechanism reliably rejects less extreme harmful repairs and behaves consistently across different missingness levels and candidate recovery strategies.

### Next Research Step

Evaluate safety acceptance and rejection decisions for zero, median, and regression-based imputation across repeated missingness injection seeds. Then investigate whether the current safety thresholds adequately identify realistic performance deterioration.

In [38]:
metrics = ["f1", "recall", "roc_auc"]
categories = ["F1", "Recall", "ROC-AUC"]
fig = go.Figure()
for name, scores in [("Healthy baseline", healthy_metrics), ("Harmful candidate", harmful_metrics)]:
    fig.add_bar(
        x=categories, y=[scores[metric] for metric in metrics], name=name,
        texttemplate="%{y:.3f}", textposition="outside",
        hovertemplate="%{x}<br>%{fullData.name}: %{y:.4f}<extra></extra>",
    )
f1_loss = (healthy_metrics["f1"] - harmful_metrics["f1"]) / healthy_metrics["f1"]
recall_loss = (healthy_metrics["recall"] - harmful_metrics["recall"]) / healthy_metrics["recall"]
fig.update_layout(
    title=("SHML Safety Validation — Harmful Candidate Rejected"
           f"<br><sup>Relative losses: F1 {f1_loss:.2%} · Recall {recall_loss:.2%}</sup>"),
    yaxis=dict(title="Metric score", range=[0, 0.85]), barmode="group",
)
show_research_figure(fig, "safety_harmful_candidate_rejection.html")


Saved: /Users/divya/Desktop/School/IU/GitStuff/SHML/results/research/bill_amt1/figures/safety_harmful_candidate_rejection.html


## Phase E — Step 11: Safety Evaluation Across Recovery Strategies

In [39]:

from pathlib import Path
import sys

# Find the SHML project root
cwd = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        p for p in (cwd, *cwd.parents)
        if (p / "src" / "evaluation.py").exists()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"SHML project root not found from {cwd}. "
        "Open the notebook inside your SHML repository."
    )

# Add project root to Python's import path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

# Verify imports
from src.evaluation import evaluate_model
from src.safety import evaluate_recovery_safety

print("SHML imports successful!")


Project root: /Users/divya/Desktop/School/IU/GitStuff/SHML
SHML imports successful!


In [40]:
from src.evaluation import evaluate_model
from src.safety import evaluate_recovery_safety

In [41]:

# Verify that all Step 11 dependencies are available

required_names = [
    "X_train",
    "X_val",
    "y_val",
    "baseline_model",
    "scaler",
    "imputation_model",
    "inject_bill_amt1_missingness",
    "recover_bill_amt1_regression"
]

missing = [
    name for name in required_names
    if name not in globals()
]

if missing:
    print("Missing variables/functions:", missing)
else:
    print("All dependencies initialized!")
    print("Training rows:", len(X_train))
    print("Validation rows:", len(X_val))
    print("BILL_AMT1 median:", X_train["BILL_AMT1"].median())


All dependencies initialized!
Training rows: 21000
Validation rows: 4500
BILL_AMT1 median: 22225.5


In [42]:

# Phase E — Step 11A
# Evaluate safety decisions across 3 recovery strategies
# and 20 missingness injection seeds.

import pandas as pd
import numpy as np

from src.evaluation import evaluate_model
from src.safety import evaluate_recovery_safety

training_median = X_train["BILL_AMT1"].median()

healthy_metrics = evaluate_model(
    X_batch=X_val,
    y_true=y_val,
    model=baseline_model,
    scaler=scaler
)

safety_results = []

for seed in range(20):
    for rate in [0.00, 0.05, 0.15, 0.30, 0.60]:

        X_corrupted, mask = inject_bill_amt1_missingness(
            X_val,
            missingness_rate=rate,
            seed=seed
        )

        # Create the three candidate repairs
        X_zero = X_corrupted.copy(deep=True)
        X_zero["BILL_AMT1"] = (
            X_zero["BILL_AMT1"].fillna(0)
        )

        X_median = X_corrupted.copy(deep=True)
        X_median["BILL_AMT1"] = (
            X_median["BILL_AMT1"].fillna(training_median)
        )

        X_regression = recover_bill_amt1_regression(
            X_corrupted,
            imputation_model
        )

        candidates = {
            "Zero": X_zero,
            "Median": X_median,
            "Regression": X_regression
        }

        for strategy, X_candidate in candidates.items():

            # Verify complete and targeted repair
            integrity_passed = (
                X_candidate.shape == X_val.shape
                and X_candidate.columns.equals(X_val.columns)
                and not X_candidate.isna().any().any()
                and X_candidate.drop(
                    columns=["BILL_AMT1"]
                ).equals(
                    X_val.drop(columns=["BILL_AMT1"])
                )
                and np.array_equal(
                    X_candidate.loc[
                        ~mask, "BILL_AMT1"
                    ].to_numpy(),
                    X_val.loc[
                        ~mask, "BILL_AMT1"
                    ].to_numpy()
                )
            )

            metrics = evaluate_model(
                X_batch=X_candidate,
                y_true=y_val,
                model=baseline_model,
                scaler=scaler
            )

            safety = evaluate_recovery_safety(
                recovered_metrics=metrics,
                healthy_metrics=healthy_metrics,
                data_integrity_passed=integrity_passed,
                max_f1_loss=0.10,
                max_recall_loss=0.10
            )

            safety_results.append({
                "Seed": seed,
                "Missingness": rate,
                "Strategy": strategy,
                "F1": metrics["f1"],
                "Recall": metrics["recall"],
                "ROC_AUC": metrics["roc_auc"],
                "Integrity_Passed": integrity_passed,
                "Decision": safety["decision"],
                "Reason": safety["reason"],
                "F1_Loss_Pct": safety["f1_loss_pct"],
                "Recall_Loss_Pct": safety["recall_loss_pct"]
            })

safety_df = pd.DataFrame(safety_results)

assert len(safety_df) == 300
assert safety_df["Integrity_Passed"].all()

print("Total candidate evaluations:", len(safety_df))
display(safety_df.head(10).round(6))


Total candidate evaluations: 300


,Seed,Missingness,Strategy,F1,Recall,ROC_AUC,Integrity_Passed,Decision,Reason,F1_Loss_Pct,Recall_Loss_Pct
0,0,0.00,Zero,0.356983,0.240201,0.713025,True,ACCEPT,All checks passed,0.000000,0.000000
1,0,0.00,Median,0.356983,0.240201,0.713025,True,ACCEPT,All checks passed,0.000000,0.000000
2,0,0.00,Regression,0.356983,0.240201,0.713025,True,ACCEPT,All checks passed,0.000000,0.000000
3,0,0.05,Zero,0.359584,0.243216,0.711930,True,ACCEPT,All checks passed,-0.728643,-1.255230
4,0,0.05,Median,0.359851,0.243216,0.711753,True,ACCEPT,All checks passed,-0.803534,-1.255230
5,0,0.05,Regression,0.356983,0.240201,0.712837,True,ACCEPT,All checks passed,0.000000,0.000000
6,0,0.15,Zero,0.372807,0.256281,0.708495,True,ACCEPT,All checks passed,-4.432761,-6.694561
7,0,0.15,Median,0.370588,0.253266,0.709936,True,ACCEPT,All checks passed,-3.811223,-5.439331
8,0,0.15,Regression,0.357250,0.240201,0.713042,True,ACCEPT,All checks passed,-0.074738,0.000000
9,0,0.30,Zero,0.382395,0.266332,0.705978,True,ACCEPT,All checks passed,-7.118706,-10.878661


In [43]:

# Phase E — Step 11B
# Summarize safety decisions

safety_summary = (
    safety_df
    .groupby(["Missingness", "Strategy"])
    .agg(
        Evaluations=("Decision", "size"),
        Accepted=(
            "Decision",
            lambda x: (x == "ACCEPT").sum()
        ),
        Rejected=(
            "Decision",
            lambda x: (x == "REJECT").sum()
        ),
        Mean_F1=("F1", "mean"),
        Mean_Recall=("Recall", "mean"),
        Mean_AUC=("ROC_AUC", "mean")
    )
    .reset_index()
)

safety_summary["Acceptance_Rate_Pct"] = (
    100 * safety_summary["Accepted"]
    / safety_summary["Evaluations"]
)

display(safety_summary.round(4))


,Missingness,Strategy,Evaluations,Accepted,Rejected,Mean_F1,Mean_Recall,Mean_AUC,Acceptance_Rate_Pct
0,0.00,Median,20,20,0,0.3570,0.2402,0.7130,100.0
1,0.00,Regression,20,20,0,0.3570,0.2402,0.7130,100.0
2,0.00,Zero,20,20,0,0.3570,0.2402,0.7130,100.0
3,0.05,Median,20,20,0,0.3593,0.2425,0.7120,100.0
4,0.05,Regression,20,20,0,0.3569,0.2401,0.7130,100.0
5,0.05,Zero,20,20,0,0.3608,0.2441,0.7117,100.0
6,0.15,Median,20,20,0,0.3648,0.2479,0.7112,100.0
7,0.15,Regression,20,20,0,0.3573,0.2404,0.7129,100.0
8,0.15,Zero,20,20,0,0.3683,0.2519,0.7107,100.0
9,0.30,Median,20,20,0,0.3686,0.2524,0.7096,100.0



### Observation E.2 — Safety Validation Across Recovery Strategies

The SHML safety controller was evaluated using three
recovery strategies: zero, median, and regression
imputation.

Experiments covered five BILL_AMT1 missingness levels:
0%, 5%, 15%, 30%, and 60%.

Each scenario was repeated across 20 injection seeds,
producing 300 candidate evaluations.

#### Results

- Total evaluations: 300
- Accepted candidates: 300
- Rejected candidates: 0
- Overall acceptance rate: 100%

All three recovery strategies satisfied the existing
safety constraints across the tested experiments.

The controller applies a maximum allowable relative
loss of 10% in F1 and recall compared with the
healthy validation baseline.

#### Interpretation

The results show that all tested recovery candidates
remained within the configured safety tolerances.

However, safety acceptance does not demonstrate
that a candidate improves predictive performance
or outperforms other recovery methods.

Earlier harmful-candidate testing demonstrated that
the controller can reject severe degradation.

Together, these experiments provide evidence of
acceptance and rejection behavior under the tested
conditions, but not universal reliability.

#### Limitations

The 20 random seeds vary missingness injection patterns
while keeping the trained model and validation data fixed.

The experiments do not yet establish rejection
sensitivity for realistic harmful repairs.

Additional threshold-sensitivity experiments and
less extreme failure scenarios are needed.
